# In this notebook we do soome data exploration and fit 2 simple models
- this is the exploration phase and simple fitting of models
- we evaluate the models in the "modeling.ipynb" notebook

In [ ]:
# ===============================================
# BASIC SETUP: GIT Auth and mounting google drive
# ================================================

from google.colab import drive, userdata
import sys

# Standard mount to access the config file initially
drive.mount('/content/drive')
PROJECT_ROOT = userdata.get('BIG_DATA_PATH')
sys.path.append(PROJECT_ROOT)

from utils.config import initialize_project
initialize_project()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# data inspection
df = pd.read_csv("data/raw/datatraining.txt")
df.head()

In [ ]:
# number of rows and columns
df.shape

In [ ]:
# checking for column types and null values
df.info()

In [ ]:
# check of null values
df.isnull().sum()

In [ ]:
# checking for duplicate values
df.duplicated().sum()

In [ ]:
df["Occupancy"].value_counts()
df["Occupancy"].value_counts(normalize=True)

In [ ]:
sns.countplot(x="Occupancy", data=df)
plt.title("Occupancy Distribution")
plt.show()

In [ ]:
df.describe() # summary statistics

In [ ]:
df.groupby("Occupancy")[["Temperature", "Humidity", "Light", "CO2", "HumidityRatio"]].mean()

In [ ]:
df[["Temperature", "Humidity", "Light", "CO2", "HumidityRatio"]].hist(figsize=(12, 8))
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df.drop(columns=["date"]).corr(), annot=True, cmap="coolwarm")
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x="Occupancy", y="Light", data=df)
plt.title("Light by Occupancy")
plt.show()

In [ ]:
features = ["Temperature", "Humidity", "Light", "CO2", "HumidityRatio"]
target = "Occupancy"
X = df[features]
y = df[target]

In [ ]:
# spliting the data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Training the first baseline model using Logistic Regression
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

In [ ]:
# make predicitions
y_pred = model.predict(X_test)


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

The model achieves high accuracy, indicating that environmental sensor data is predictive of room occupancy. The model performs particularly well on detecting occupied rooms, likely due to strong signals from light and CO₂ levels.

In [ ]:
# identifying the most important features for prediction
coefficients = pd.DataFrame({
    "Feature": features,
    "Coefficient": model.coef_[0]
}).sort_values(by="Coefficient", ascending=False)

coefficients

In [ ]:
# in the previous model we didnt scale our features, each feature had its own scale
# when dealing with logistic regression all features in the dataset must be on the same scale for accurate results
# here we are now scalling the features and then re-fitting the model
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

In [ ]:
# identifying the most important features for prediction
coefficients = pd.DataFrame({
    "Feature": features,
    "Coefficient": model.coef_[0]
}).sort_values(by="Coefficient", ascending=False)

coefficients

from this we can conclude that light and CO2 are the most important predictors for occupancy in a room

In [ ]:
# Random Forest Model
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("RF Accuracy:", accuracy_score(y_test, rf_pred))
print(classification_report(y_test, rf_pred))

In [ ]:
# Comparing the Models
results = {
    "Logistic Regression": accuracy_score(y_test, y_pred),
    "Random Forest": accuracy_score(y_test, rf_pred)
}

results

Random Forest outperformed Logistic Regression, indicating nonlinear relationships between environmental features and occupancy.


In [ ]:
importance = pd.DataFrame({
    "Feature": features,
    "Importance": rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

importance

We extract the most important features for prediction for the random forest model and we can see that the most important features are:


1.   Light
2.   C02
3.   temperature

